In [ ]:
import os 
from tqdm import tqdm
import pandas as pd 
import numpy as np

args = {
'dataset' : 'synthetic', 
'entity' : 'point_context', # Which entity within the dataset to evaluate, mixed. Default options: "mixed", "shapelet", "seasonal", "trend", "point_context", "point_global"
'model' : 'ae', # Model to evaluate, string. Default options: 'ae', 'vae', 'lstmvae', 'pca',
'hidden_dim' : 'default', # Default option: 'default' to use model parameters specified in paper, or integer
'seq_len' : 12, # Length of the input window, integer. Default option: 12
'num_epochs' : 100, # Number of training epochs, integer. Default: 100
'verbose' : False
}
val_split = 0.1 
batch_size = 64

experiment_name = "/seq" + str(args['seq_len'])
args['experiment_dir'] = "experiments/" + args['dataset'] + experiment_name
print("Experiments will be saved to " + args['experiment_dir'])

os.makedirs(args['experiment_dir'], exist_ok=True) 
    
from utils.preprocess import get_data, valid_entities
from utils.utils import seq_data, save_metrics
from torch.utils.data import DataLoader

entity = args['entity']
if entity == 'none':
    entity = None 

#Load data
train_data, val_data, test_data, test_labels = get_data(args['dataset'], val_split = val_split, 
                                                        seq_len = args['seq_len'], 
                                                        entity = entity, verbose = args['verbose'])
    
input_dim = train_data.shape[-1]
args['input_dim'] = input_dim
print(input_dim)
    
train_dataset = seq_data(train_data, args['seq_len'])
val_dataset = seq_data(val_data, args['seq_len'])
test_dataset = seq_data(test_data, args['seq_len'], test_labels) 

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size)

def save_result(results_dict, metrics_df, verbose = True):    
    metrics_df = pd.concat([metrics_df, pd.DataFrame(results_dict)], ignore_index = True, join = "outer")
    metrics_df.drop_duplicates(inplace = True)
#     metrics_df.to_csv(args['experiment_dir'] + "/metrics.csv", index = False)
    
#     if verbose:
#         print(metrics_df.tail(5))
    return metrics_df


In [ ]:
from models.AE import train_and_calibrate_ae
from models.VAE import train_and_calibrate_vae
from models.LSTMVAE import train_and_calibrate_lstmvae
from models.PCA import train_and_calibrate_pca 


#Stores and keeps track of possible experiments 
model_zoo = {
    'ae': train_and_calibrate_ae, 
    'vae': train_and_calibrate_vae, 
    'lstmvae': train_and_calibrate_lstmvae,   
    'pca': train_and_calibrate_pca, 
}


In [ ]:
if args['model'] == 'pca':
    scores, labels, latents, calibration_nc, calibration_latents = model_zoo[args['model']](train_data, val_data, test_data, test_labels, args)
else:
    scores, labels, latents, calibration_nc, calibration_latents = model_zoo[args['model']](train_loader, val_loader, test_loader, args)

# Conformal Thresholding

In [ ]:
from scipy.spatial.distance import cdist
import math

epsilon = 0.05
gamma = 0.05

def compute_threshold(scores, alpha):
    threshold = np.quantile(scores, 1-alpha)
    return threshold

high_thresh = compute_threshold(calibration_nc,epsilon) #normal calibration baseline(doesnt change)
w_thresh = [] #weighted calibration
wa_thresh = [] #weighted calibration + adjustment 
a_thresh = [] #normal calibration + adjustment 

adjusts = []
min_distances = []

for i, nc in enumerate(tqdm(scores)):
    distances = cdist(latents[i].reshape((1, -1)),calibration_latents, metric = 'sqeuclidean')[0]
    
    #quantile adjustment
    adjust = 2*gamma*math.tanh(np.min(distances))-gamma
    adjusts += [adjust]
    
    #weighting
    distances = 1-(distances/distances.sum())
    w_calibration = distances * calibration_nc

    w_thresh.append(compute_threshold(w_calibration, epsilon))    
    wa_thresh.append(compute_threshold(w_calibration,epsilon+adjust))
    a_thresh.append(compute_threshold(calibration_nc,epsilon+adjust))   
    

# Metrics 

In [ ]:
metrics_dict = {
'model' : [args['model']],
'entity' : [args['entity']]}

if os.path.isfile(args['experiment_dir'] + "/metrics.csv"):
    metrics_df = pd.read_csv(args['experiment_dir'] + "/metrics.csv")
    print("reading previously saved metrics")
else:    
    metrics_df = pd.DataFrame(columns = ["model", "entity", "method", "score", "tn", "fn", "fp", "tp"])
    print("starting new metrics records")    


print(metrics_dict)
metrics_df.tail(5)

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix
from utils.utils import ROC

def thresholding(scores, labels, thresholds, metrics_dict):
    thresholded = []     
    if len(thresholds) == 1: 
        for i in range(len(scores)):
            thresholded.append(1 if scores[i] > thresholds else 0)            
    else:
        for i in range(len(scores)):
            thresholded.append(1 if scores[i] > thresholds[i] else 0)
            
    tn, fp, fn, tp = confusion_matrix(labels, thresholded).ravel().tolist()
    n = len(labels)
    metrics_dict['tn'] = tn/n
    metrics_dict['fn'] = fn/n
    metrics_dict['fp'] = fp/n
    metrics_dict['tp'] = tp/n
    metrics_dict['score'] = f1_score(labels, thresholded)
    
    return thresholded, metrics_dict

In [ ]:
#Best-F1 
metrics_dict['method'] = ["best F1"]
thresh, auc = ROC(labels, scores)
thresholded, metrics_dict = thresholding(scores, labels, [thresh], metrics_dict)

print(f'Best F1:{metrics_dict["score"]:.3f}')
print(f'TN: {metrics_dict["tn"]:.3f}, FN: {metrics_dict["fn"]:.3f}, FP: {metrics_dict["fp"]:.3f}, TP: {metrics_dict["tp"]:.3f}')
metrics_df = save_result(metrics_dict, metrics_df)

#0.05
metrics_dict['method'] = ["sig-0.05"]
thresholded, metrics_dict = thresholding(scores, labels, [high_thresh], metrics_dict)
print(f'Basic Conformal F1: {metrics_dict["score"]:.3f}')
print(f'TN: {metrics_dict["tn"]:.3f}, FN: {metrics_dict["fn"]:.3f}, FP: {metrics_dict["fp"]:.3f}, TP: {metrics_dict["tp"]:.3f}')
metrics_df = save_result(metrics_dict, metrics_df)

#weighted
metrics_dict['method'] = ["weighted"]
thresholded, metrics_dict = thresholding(scores, labels, w_thresh, metrics_dict)
print(f'Weighted F1: {metrics_dict["score"]:.3f}')
print(f'TN: {metrics_dict["tn"]:.3f}, FN: {metrics_dict["fn"]:.3f}, FP: {metrics_dict["fp"]:.3f}, TP: {metrics_dict["tp"]:.3f}')
metrics_df = save_result(metrics_dict, metrics_df)

#Adjusted
metrics_dict['method'] = ["adjusted"]
thresholded, metrics_dict = thresholding(scores, labels, a_thresh, metrics_dict)
print(f'Adjusted F1: {metrics_dict["score"]:.3f}')
print(f'TN: {metrics_dict["tn"]:.3f}, FN: {metrics_dict["fn"]:.3f}, FP: {metrics_dict["fp"]:.3f}, TP: {metrics_dict["tp"]:.3f}')
metrics_df = save_result(metrics_dict, metrics_df)

#Weighted + adjusted 
metrics_dict['method'] = ["weighted+adjusted"]
thresholded, metrics_dict = thresholding(scores, labels, wa_thresh, metrics_dict)
print(f'Weighted Adjusted F1: {metrics_dict["score"]:.3f}')
print(f'TN: {metrics_dict["tn"]:.3f}, FN: {metrics_dict["fn"]:.3f}, FP: {metrics_dict["fp"]:.3f}, TP: {metrics_dict["tp"]:.3f}')
metrics_df = save_result(metrics_dict, metrics_df)

In [ ]:
metrics_df.to_csv(args['experiment_dir'] + "/metrics.csv", index = False)

# Synthetic Dataset Generation 
Code from https://github.com/alexcapstick/tods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def series_segmentation(data, stepsize=1):
    return np.split(data, np.where(np.diff(data) != stepsize)[0] + 1)


def sine(length, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    # timestamp = np.linspace(0, 10, length)
    timestamp = np.arange(length)
    value = np.sin(2 * np.pi * freq * timestamp)
    if noise_amp != 0:
        noise = np.random.normal(0, 1, length)
        value = value + noise_amp * noise
    value = coef * value + offset
    return value

def cosine(length, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    # timestamp = np.linspace(0, 10, length)
    timestamp = np.arange(length)
    value = np.cos(2 * np.pi * freq * timestamp)
    if noise_amp != 0:
        noise = np.random.normal(0, 1, length)
        value = value + noise_amp * noise
    value = coef * value + offset
    return value


def square_sine(level=5, length=500, freq=0.04, coef=1.5, offset=0.0, noise_amp=0.05):
    value = np.zeros(length)
    for i in range(level):
        value += 1 / (2 * i + 1) * sine(length=length, freq=freq * (2 * i + 1), coef=coef, offset=offset, noise_amp=noise_amp)
    return value


def collective_global_synthetic(length, base, coef=1.5, noise_amp=0.005):
    value = []
    norm = np.linalg.norm(base)
    base = base / norm
    num = int(length / len(base))
    for i in range(num):
        value.extend(base)
    residual = length - len(value)
    value.extend(base[:residual])
    value = np.array(value)
    noise = np.random.normal(0, 1, length)
    value = coef * value + noise_amp * noise
    return value


class MultivariateDataGenerator:
    def __init__(self, dim, stream_length, behavior, behavior_config=None):

        self.dim = dim
        self.STREAM_LENGTH = stream_length
        self.behavior = behavior if behavior is not None else [sine] * dim
        self.behavior_config = behavior_config if behavior_config is not None else [{}] * dim

        self.data = np.empty(shape=[0, stream_length],dtype=float)
        self.label = None
        self.data_origin = None
        self.timestamp = np.arange(self.STREAM_LENGTH)

        self.generate_timeseries()

    def generate_timeseries(self):
        for i in range(self.dim):
            self.behavior_config[i]['length'] = self.STREAM_LENGTH
            self.data = np.append(self.data, [self.behavior[i](**self.behavior_config[i])], axis=0)
        self.data_origin = self.data.copy()
        self.label = np.zeros(self.STREAM_LENGTH, dtype=int)

    def point_global_outliers(self, dim_no, ratio, factor, radius, position = None):
        """
        Add point global outliers to original data
        Args:
            ratio: what ratio outliers will be added
            factor: the larger, the outliers are farther from inliers
            radius: the radius of collective outliers range
        """
        if position == None:
            position = (np.random.rand(round(self.STREAM_LENGTH * ratio)) * self.STREAM_LENGTH).astype(int)
            
        maximum, minimum = max(self.data[dim_no]), min(self.data[dim_no])
        for i in position:
            local_std = self.data_origin[dim_no][max(0, i - radius):min(i + radius, self.STREAM_LENGTH)].std()
            self.data[dim_no][i] = self.data_origin[dim_no][i] * factor * local_std
            if 0 <= self.data[dim_no][i] < maximum: self.data[dim_no][i] = maximum
            if 0 > self.data[dim_no][i] > minimum: self.data[dim_no][i] = minimum
            self.label[i] = 1

    def point_contextual_outliers(self,dim_no, ratio, factor, radius, position = None):
        """
        Add point contextual outliers to original data
        Args:
            ratio: what ratio outliers will be added
            factor: the larger, the outliers are farther from inliers
                    Notice: point contextual outliers will not exceed the range of [min, max] of original data
            radius: the radius of collective outliers range
        """
        if position == None:
            position = (np.random.rand(round(self.STREAM_LENGTH * ratio)) * self.STREAM_LENGTH).astype(int)
        maximum, minimum = max(self.data[dim_no]), min(self.data[dim_no])
        for i in position:
            local_std = self.data_origin[dim_no][max(0, i - radius):min(i + radius, self.STREAM_LENGTH)].std()
            self.data[dim_no][i] = self.data_origin[dim_no][i] * factor * local_std
            if self.data[dim_no][i] > maximum: self.data[dim_no][i]= maximum * min(0.95, abs(np.random.normal(0, 1)))
            if self.data[dim_no][i] < minimum: self.data[dim_no][i] = minimum * min(0.95, abs(np.random.normal(0, 1)))

            self.label[i] = 1

    def collective_global_outliers(self,dim_no, ratio, radius, option='square', coef=3., noise_amp=0.0,
                                    level=5, freq=0.04, offset=0.0, # only used when option=='square'
                                    base=[0.,], position = None): # only used when option=='other'
        """
        Add collective global outliers to original data
        Args:
            ratio: what ratio outliers will be added
            radius: the radius of collective outliers range
            option: if 'square': 'level' 'freq' and 'offset' are used to generate square sine wave
                    if 'other': 'base' is used to generate outlier shape
            level: how many sine waves will square_wave synthesis
            base: a list of values that we want to substitute inliers when we generate outliers
        """
        if position == None:
            position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)

        valid_option = {'square', 'other'}
        if option not in valid_option:
            raise ValueError("'option' must be one of %r." % valid_option)

        if option == 'square':
            sub_data = square_sine(level=level, length=self.STREAM_LENGTH, freq=freq,
                                   coef=coef, offset=offset, noise_amp=noise_amp)
        else:
            sub_data = collective_global_synthetic(length=self.STREAM_LENGTH, base=base,
                                                   coef=coef, noise_amp=noise_amp)
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            self.data[dim_no][start:end] = sub_data[start:end]
            self.label[start:end] = 1

    def collective_trend_outliers(self,dim_no, ratio, factor, radius, position = None):
        """
        Add collective trend outliers to original data
        Args:
            ratio: what ratio outliers will be added
            factor: how dramatic will the trend be
            radius: the radius of collective outliers range
        """
        if position == None:
            position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            slope = np.random.choice([-1, 1]) * factor * np.arange(end - start)
            self.data[dim_no][start:end] = self.data_origin[dim_no][start:end] + slope
            self.data[dim_no][end:] = self.data[dim_no][end:] + slope[-1]
            self.label[start:end] = 1

    def collective_seasonal_outliers(self,dim_no, ratio, factor, radius, position = None):
        """
        Add collective seasonal outliers to original data
        Args:
            ratio: what ratio outliers will be added
            factor: how many times will frequency multiple
            radius: the radius of collective outliers range
        """
        if position == None:
            position = (np.random.rand(round(self.STREAM_LENGTH * ratio / (2 * radius))) * self.STREAM_LENGTH).astype(int)
        seasonal_config = self.behavior_config[dim_no]
        seasonal_config['freq'] = factor * self.behavior_config[dim_no]['freq']
        for i in position:
            start, end = max(0, i - radius), min(self.STREAM_LENGTH, i + radius)
            self.data[dim_no][start:end] = self.behavior[dim_no](**seasonal_config)[start:end]
            self.label[start:end] = 1

In [ ]:
np.random.seed(100)

BEHAVIOR = [sine, cosine, sine, cosine, sine]
BEHAVIOR_CONFIG = [{'freq': 0.04, 'coef': 1.5, "offset": 0.0, 'noise_amp': 0.05},
                   {'freq': 0.04, 'coef': 2.5, "offset": 0.0, 'noise_amp': 0.05},
                   {'freq': 0.04, 'coef': 1.5, "offset": 0.0, 'noise_amp': 0.1},
                   {'freq': 0.04, 'coef': 2.5, "offset": 0.0, 'noise_amp': 0.1},
                   {'freq': 0.04, 'coef': 1.5, "offset": -2.0, 'noise_amp': 0.05},]

multivariate_data = MultivariateDataGenerator(dim=5, stream_length=500, behavior=BEHAVIOR,
                                            behavior_config=BEHAVIOR_CONFIG)



# Settings used to generate anomalies  
# multivariate_data.point_global_outliers(dim_no=i, ratio=0.05, factor=3.5, radius=5)
# multivariate_data.point_contextual_outliers(dim_no=i, ratio=0.05, factor=2.5, radius=5)
# multivariate_data.collective_global_outliers(dim_no=i, ratio=0.05, radius=5, option='square', coef=1.5, noise_amp=0.03, level=20, freq=0.04, offset=0.0)
# multivariate_data.collective_seasonal_outliers(dim_no=i, ratio=0.05, factor=3, radius=5)

for i in range(5):
    multivariate_data.collective_global_outliers(dim_no=i, ratio=0.05, radius=5, option='square', coef=1.5, noise_amp=0.03, level=20, freq=0.04, offset=0.0)

df = pd.DataFrame({'col_0': multivariate_data.data[0],
                   'col_1': multivariate_data.data[1],
                   'col_2': multivariate_data.data[2],
                   'col_3': multivariate_data.data[3],
                   'col_4': multivariate_data.data[4]})


df_train = pd.DataFrame(train, columns = ['col_0','col_1','col_2','col_3','col_4'])
df_test = pd.DataFrame(test, columns = ['col_0','col_1','col_2','col_3','col_4'])
df_labels = pd.DataFrame(lables)

#saving
# df_train.to_csv("./datasets/random/train.csv", index=False)
# df_test.to_csv("./datasets/random/test.csv", index=False)
# df_labels.to_csv("./datasets/random/labels.csv", index=False)
